In [33]:
import os
import shutil
from spatialdata_io import visium
import tarfile
import pandas as pd
import matplotlib.pyplot as plt
import spatialdata_plot  # noqa: F401
import warnings
warnings.filterwarnings('ignore')

In [34]:
in_dir = "../../Data/10x_visium"
out_dir = "../../results/09_analysis_podocyte_niche/04_prepare_data"

os.makedirs(out_dir, exist_ok=True)

In [35]:
sample_list = ['B23-05785-7-1', 'B23-05785-7-2', 'X1845']

In [36]:
for sample in sample_list:
    if sample == 'X1845':
        df = pd.read_csv(f"{in_dir}/{sample}/spatial/tissue_positions_old.csv")
        # mirror the coordinates across the vertical axis
        _max = df['pxl_col_in_fullres'].max()
        df['pxl_col_in_fullres'] = _max - df['pxl_col_in_fullres']

        _max = df['array_col'].max()
        df['array_col'] = _max - df['array_col']

        df.to_csv(f"{in_dir}/{sample}/spatial/tissue_positions.csv", 
                  index=False)

        sdata = visium(path=f"{in_dir}/{sample}",
                    dataset_id=f"{sample}", 
                    fullres_image_file=f'{sample}.tif',
                    var_names_make_unique=True)
    else:
        sdata = visium(path=f"{in_dir}/{sample}",
                    dataset_id=f"{sample}", 
                    fullres_image_file=f'{sample}.tif',
                    var_names_make_unique=True)

    # visualize the data to check if the shapes are correctly 
    # loaded and aligned with the images
    fig, ax = plt.subplots(figsize=(8, 8), dpi=300)
    (
        sdata.pl.render_images(f"{sample}_full_image")
        .pl.render_shapes(sample, color="POSTN")
        .pl.show(coordinate_systems=sample, ax=ax, 
                 title=f"{sample} - POSTN expression")
    )

    fig.savefig(f"{out_dir}/{sample}_POSTN.png", 
                dpi=300, bbox_inches="tight")
    plt.close()

    if os.path.exists(f"{out_dir}/{sample}.zarr"):
        shutil.rmtree(f"{out_dir}/{sample}.zarr")

    sdata.write(f"{out_dir}/{sample}.zarr")

    with tarfile.open(f"{out_dir}/{sample}.zarr.tar.gz", "w:gz") as tar:
        tar.add(f"{out_dir}/{sample}.zarr", arcname=f"{sample}.zarr")

    # remove the uncompressed zarr directory
    shutil.rmtree(f"{out_dir}/{sample}.zarr")

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.11764706..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.13736264..1.0].
